<a href="https://colab.research.google.com/github/CodeJamjamzz/Intelligent_Systems_VAE_Project/blob/main/CartoonFace_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kmader/cartoon-set")

print("Path to dataset files:", path)

100%|██████████| 4.92G/4.92G [00:58<00:00, 90.7MB/s]

Extracting files...


In [ ]:
import os
from pathlib import Path
# Use pathlib to recursively find all .png files
dataset_path = Path(path)
image_paths = list(dataset_path.rglob("*.png"))

# Convert PosixPath objects to strings for easier handling later
image_paths = [str(p) for p in image_paths]

print(f"Successfully found {len(image_paths)} images.")
print("Example path:", image_paths[0])

In [ ]:
import tensorflow as tf

# Convert the list of paths into a tf.data.Dataset
path_ds = tf.data.Dataset.from_tensor_slices(image_paths)

def load_and_preprocess_image(file_path):
    # Read the file
    img = tf.io.read_file(file_path)
    # Decode the PNG (channels=3 ignores alpha channel)
    img = tf.image.decode_png(img, channels=3)
    # Resize for the VAE bottleneck
    img = tf.image.resize(img, [64, 64])
    # Normalize pixels to [0, 1]
    img = img / 255.0
    return img

# Map the loading function to all paths
image_ds = path_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Shuffle, batch, and prefetch for performance
BATCH_SIZE = 64
train_dataset = (
    image_ds
    .shuffle(buffer_size=1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Test the loader
for batch in train_dataset.take(1):
    print("Batch shape:", batch.shape) # Expected: (64, 64, 64, 3)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
latent_dim = 128 # The size of the compressed representation

# ==========================================
# ENCODER
# ==========================================
encoder_inputs = keras.Input(shape=(64, 64, 3))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2D(128, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2D(256, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)

# The encoder outputs the mean and log variance for the latent distribution
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

# ==========================================
# DECODER
# ==========================================
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(4 * 4 * 256, activation="relu")(latent_inputs)
x = layers.Reshape((4, 4, 256))(x) # Reshape to match the encoder's pre-flatten shape

# Transposed convolutions to upsample back to 64x64
x = layers.Conv2DTranspose(256, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(128, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)

# Final layer uses a sigmoid activation because image pixels are normalized to [0, 1]
decoder_outputs = layers.Conv2DTranspose(3, 3, activation="sigmoid", padding="same")(x)

decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

        # Trackers for the metrics
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            # 1. Pass data through encoder to get latent space distributions
            z_mean, z_log_var, z = self.encoder(data)

            # 2. Pass sampled latent vector through decoder to reconstruct the image
            reconstruction = self.decoder(z)

            # 3. Calculate Reconstruction Loss (Binary Crossentropy is standard for normalized images)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction),
                    axis=(1, 2)
                )
            )

            # 4. Calculate KL Divergence Loss
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))

            # 5. Total Loss
            total_loss = reconstruction_loss + kl_loss

        # Compute gradients and update weights
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
# Instantiate the VAE
vae = VAE(encoder, decoder)

# Compile using Adam optimizer (learning rate is generally kept low for VAEs)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005))

# Assuming 'train_dataset' is the tf.data.Dataset from the previous data pipeline
vae.fit(train_dataset, epochs=30)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def latent_space_interpolation(model, dataset, steps=10):
    # 1. Grab two different images from your dataset
    for batch in dataset.take(1):
        img_a = batch[0:1] # Person A
        img_b = batch[1:2] # Person B
        break

    # 2. Encode them to find their "coordinates" (z) in the latent space
    # We use the mean (z_mean) for the most stable representation
    z_mean_a, _, _ = model.encoder.predict(img_a)
    z_mean_b, _, _ = model.encoder.predict(img_b)

    # 3. Create a linear path between point A and point B
    # This creates 'steps' number of coordinates along a straight line
    alphas = np.linspace(0, 1, steps)
    interpolated_z = np.array([(1 - a) * z_mean_a + a * z_mean_b for a in alphas])

    # Reshape to (steps, latent_dim) for the decoder
    interpolated_z = interpolated_z.reshape(steps, -1)

    # 4. Decode these coordinates back into images
    decoded_images = model.decoder.predict(interpolated_z)

    # 5. Visualize the "Morph"
    plt.figure(figsize=(20, 4))
    for i in range(steps):
        ax = plt.subplot(1, steps, i + 1)
        plt.imshow(decoded_images[i])
        plt.axis("off")
        if i == 0: plt.title("Start (A)")
        if i == steps // 2: plt.title("Hybrid")
        if i == steps - 1: plt.title("End (B)")

    plt.show()

# Run the test
latent_space_interpolation(vae, train_dataset)